# Day 1: I Own the Training Loop

**Goal**: Train an NLP model from scratch and debug it confidently.

**Time**: 4 hours
- 20 min: Planning
- 2.5 hrs: Coding
- 45 min: Debugging & experiments
- 25 min: Reflection

---

## Confidence Checks
- [ ] I can explain why incorrect padding or masking breaks training
- [ ] I can change embedding size or max sequence length without panic
- [ ] I know exactly where NaNs come from

## Task 1: Text Pipeline (45 min)

**Dataset**: IMDb (5k samples)

Build:
- Whitespace tokenizer
- Vocabulary (PAD=0, UNK=1)
- Padding and attention masks
- Custom Dataset and DataLoader

In [ ]:
# Imports
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchmetrics

# Import everything from nlp_utils package
from nlp_utils import (
    tokenize,
    Vocabulary,
    ReviewDataSet,
    collate_fn,
    extract_imdb_sample_as_dict,
    SentimentModel,
    Trainer
)

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [2]:
from datasets import load_dataset

In [3]:
ds = load_dataset("stanfordnlp/imdb")

In [4]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [5]:
# Extract sampled splits as dictionaries
train, test = extract_imdb_sample_as_dict(ds, train_size=5000, test_size=1000)

In [6]:
type(train)

dict

In [7]:
sample_text = "Hello! This is a simple example. Let's tokenize this text."
    
tokens = tokenize(sample_text)

In [8]:
# Create and build vocabulary
vocab = Vocabulary(tokenizer=tokenize)
vocab.build_from_texts(train['text'])

Vocabulary size: 38553


In [9]:
# Create datasets
train_dataset = ReviewDataSet(target=train, vocab=vocab)
test_dataset = ReviewDataSet(target=test, vocab=vocab)

In [10]:
# Create DataLoaders
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,          # Shuffle for training
    collate_fn=collate_fn,
    num_workers=0          # Set to 0 for debugging, increase for speed
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,         # Don't shuffle for testing
    collate_fn=collate_fn,
    num_workers=0
)




In [11]:
batch=next(iter(train_loader))

In [12]:
vocab.get_vocab_size()

38553

In [32]:
vocab.pad_token_id

0

# Training

## Task 2: Training Loop (90 min)

**Build from scratch with validation after every epoch**

In [ ]:
# Model is now in nlp_utils package!
# No need to define it here - just import

In [49]:
model=SentimentModel(vocab=vocab)
output=model(batch)

In [50]:
output.shape

torch.Size([32, 512, 2])

In [ ]:
# Trainer is now in nlp_utils package!
# No need to define it here - just import

In [ ]:
# Initialize model and optimizer
model = SentimentModel(vocab=vocab, embedding_dim=256, hidden_dim=128, num_categories=2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

# Define metrics using torchmetrics
metrics = {
    'acc': torchmetrics.Accuracy(task='multiclass', num_classes=2),
    'f1': torchmetrics.F1Score(task='multiclass', num_classes=2)
}

# Create trainer
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device=device,
    metrics=metrics,
    grad_clip_norm=1.0
)

# Train!
history = trainer.fit(num_epochs=5)